# Train Corpus in Colab (minimal, project-reuse)

Corpus recommande: WikiText-103 (bon compromis taille/qualite pour Colab).

Ce notebook prepare les fichiers texte, les pre-tokenise vers un format compact sur disque, puis lance l'entrainement depuis les fichiers tokenises pour eviter le chargement complet en RAM.

In [ ]:
!nvidia-smi || true
!pip -q install datasets

In [ ]:
!git clone https://github.com/your-username/transformer-lab.git 2>/dev/null || true
%cd /content/transformer-lab

In [ ]:
from pathlib import Path
from datasets import load_dataset

CORPUS_DIR = Path('/content/data')
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_TEXT_PATH = CORPUS_DIR / 'wikitext103_train.txt'
VAL_TEXT_PATH = CORPUS_DIR / 'wikitext103_val.txt'
TOKEN_DIR = CORPUS_DIR / 'wikitext103_tokens'

if not TRAIN_TEXT_PATH.exists():
    ds_train = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train')
    train_text = '\n'.join([row['text'] for row in ds_train if row['text'].strip()])
    TRAIN_TEXT_PATH.write_text(train_text, encoding='utf-8')

if not VAL_TEXT_PATH.exists():
    ds_val = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation')
    val_text = '\n'.join([row['text'] for row in ds_val if row['text'].strip()])
    VAL_TEXT_PATH.write_text(val_text, encoding='utf-8')

if not (TOKEN_DIR / 'meta.json').exists():
    !python -m src.data.loader \
      --dataset_type raw_text \
      --train_text_path /content/data/wikitext103_train.txt \
      --val_text_path /content/data/wikitext103_val.txt \
      --output_dir /content/data/wikitext103_tokens

print(f'train_text_path={TRAIN_TEXT_PATH}')
print(f'val_text_path={VAL_TEXT_PATH}')
print(f'token_metadata_path={TOKEN_DIR / "meta.json"}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/transformer_checkpoints_wikitext103'

In [ ]:
!python -m src.training.trainer \
  --dataset_type raw_text \
  --token_metadata_path /content/data/wikitext103_tokens/meta.json \
  --seq_len 256 \
  --num_epochs 10 \
  --batch_size 32 \
  --d_model 384 \
  --n_heads 6 \
  --n_layers 8 \
  --learning_rate 3e-4 \
  --checkpoint_dir /content/drive/MyDrive/transformer_checkpoints_wikitext103